<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day5_4_(260528)_Spring_Boot_CRUD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%writefile /content/spring-lab/simple-crud/src/main/java/com/example/demo/domain/Member.java

package com.example.demo.domain;

import java.time.LocalDateTime;

/**
 * [Domain 계층]
 * 소프트웨어의 핵심 비즈니스 대상과 데이터베이스 테이블을 매핑하는 클래스입니다.
 * MariaDB의 'members' 테이블의 한 행(Row)과 1:1로 대응되는 객체입니다.
 */
public class Member {

    // 데이터베이스의 각 컬럼(Column)과 매핑되는 필드 정의
    private Long id;              // 회원 고유 번호 (DB의 Primary Key, 자동 증가 분)
    private String name;          // 회원의 이름
    private String email;         // 회원의 이메일 주소
    private LocalDateTime createdAt; // 회원 등록 일시 (DB의 기본값 CURRENT_TIMESTAMP 매핑)

    /**
     * [생성자 (Constructor)]
     * 데이터베이스(Repository 계층)에서 조회된 결과를 바탕으로
     * 자바 세상에서 안전하게 사용할 Member 객체를 완성할 때 사용합니다.
     */
    public Member(Long id, String name, String email, LocalDateTime createdAt) {
        this.id = id;
        this.name = name;
        this.email = email;
        this.createdAt = createdAt;
    }

    /**
     * [Getter 메서드]
     * 외부(Service 계층이나 Thymeleaf 뷰 템플릿 등)에서 회원의 정보를 안전하게 읽어갈 수 있도록 제공합니다.
     * * 💡 왜 Setter는 없을까요?
     * 도메인의 데이터가 애플리케이션 사방에서 함부로 변경(오염)되는 것을 막기 위함입니다.
     * 데이터의 무결성을 유지하기 위해 생성자로만 값을 넣고 읽기 전용(Getter)으로 설계하는 것이 관례입니다.
     */
    public Long getId() {
        return id;
    }

    public String getName() {
        return name;
    }

    public String getEmail() {
        return email;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Writing /content/spring-lab/simple-crud/src/main/java/com/example/demo/domain/Member.java


In [5]:
%%writefile /content/spring-lab/simple-crud/src/main/java/com/example/demo/dto/MemberForm.java

package com.example.demo.dto;

public class MemberForm {

    private String name;
    private String email;

    public MemberForm() {
    }

    public MemberForm(String name, String email) {
        this.name = name;
        this.email = email;
    }

    public String getName() {
        return name;
    }

    public String getEmail() {
        return email;
    }

    public void setName(String name) {
        this.name = name;
    }

    public void setEmail(String email) {
        this.email = email;
    }
}


Writing /content/spring-lab/simple-crud/src/main/java/com/example/demo/dto/MemberForm.java


In [6]:
%%writefile /content/spring-lab/simple-crud/src/main/java/com/example/demo/repository/MemberRepository.java

package com.example.demo.repository;

import com.example.demo.domain.Member;
import com.example.demo.dto.MemberForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public class MemberRepository {

    private final JdbcTemplate jdbcTemplate;

    public MemberRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<Member> memberRowMapper = (rs, rowNum) -> new Member(
            rs.getLong("id"),
            rs.getString("name"),
            rs.getString("email"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public List<Member> findAll() {
        String sql = """
                SELECT id, name, email, created_at
                FROM members
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, memberRowMapper);
    }

    public Member findById(Long id) {
        String sql = """
                SELECT id, name, email, created_at
                FROM members
                WHERE id = ?
                """;

        return jdbcTemplate.queryForObject(sql, memberRowMapper, id);
    }

    public void save(MemberForm form) {
        String sql = """
                INSERT INTO members (name, email)
                VALUES (?, ?)
                """;

        jdbcTemplate.update(sql, form.getName(), form.getEmail());
    }

    public void update(Long id, MemberForm form) {
        String sql = """
                UPDATE members
                SET name = ?, email = ?
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, form.getName(), form.getEmail(), id);
    }

    public void delete(Long id) {
        String sql = """
                DELETE FROM members
                WHERE id = ?
                """;

        jdbcTemplate.update(sql, id);
    }
}


Writing /content/spring-lab/simple-crud/src/main/java/com/example/demo/repository/MemberRepository.java


In [7]:
%%writefile /content/spring-lab/simple-crud/src/main/java/com/example/demo/service/MemberService.java


package com.example.demo.service;

import com.example.demo.domain.Member;
import com.example.demo.dto.MemberForm;
import com.example.demo.repository.MemberRepository;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class MemberService {

    private final MemberRepository repository;

    public MemberService(MemberRepository repository) {
        this.repository = repository;
    }

    public List<Member> findAllMembers() {
        return repository.findAll();
    }

    public Member findMember(Long id) {
        return repository.findById(id);
    }

    public void createMember(MemberForm form) {
        repository.save(form);
    }

    public void updateMember(Long id, MemberForm form) {
        repository.update(id, form);
    }

    public void deleteMember(Long id) {
        repository.delete(id);
    }
}

Writing /content/spring-lab/simple-crud/src/main/java/com/example/demo/service/MemberService.java


In [9]:
%%writefile /content/spring-lab/simple-crud/src/main/java/com/example/demo/controller/MemberController.java

package com.example.demo.controller;

import com.example.demo.domain.Member;
import com.example.demo.dto.MemberForm;
import com.example.demo.service.MemberService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

import java.util.List;

@Controller
public class MemberController {

    private final MemberService service;

    public MemberController(MemberService service) {
        this.service = service;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/members";
    }

    @GetMapping("/members")
    public String list(Model model) {
        List<Member> members = service.findAllMembers();

        model.addAttribute("members", members);
        model.addAttribute("memberForm", new MemberForm());

        return "members";
    }

    @PostMapping("/members")
    public String create(@ModelAttribute MemberForm form) {
        service.createMember(form);

        return "redirect:/members";
    }

    @GetMapping("/members/{id}/edit")
    public String editForm(@PathVariable Long id, Model model) {
        Member member = service.findMember(id);

        model.addAttribute("member", member);
        model.addAttribute("memberForm", new MemberForm(member.getName(), member.getEmail()));

        return "edit";
    }

    @PostMapping("/members/{id}/edit")
    public String update(
            @PathVariable Long id,
            @ModelAttribute MemberForm form
    ) {
        service.updateMember(id, form);

        return "redirect:/members";
    }

    @PostMapping("/members/{id}/delete")
    public String delete(@PathVariable Long id) {
        service.deleteMember(id);

        return "redirect:/members";
    }
}

Overwriting /content/spring-lab/simple-crud/src/main/java/com/example/demo/controller/MemberController.java


In [12]:
%%writefile /content/spring-lab/simple-crud/src/main/resources/templates/members.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>회원 관리</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container">
    <h1>회원 관리</h1>

    <section class="card">
        <h2>회원 등록</h2>

        <form action="/members" method="post" th:object="${memberForm}">
            <div class="form-row">
                <label>이름</label>
                <input type="text" th:field="*{name}" placeholder="이름 입력" required>
            </div>

            <div class="form-row">
                <label>이메일</label>
                <input type="email" th:field="*{email}" placeholder="이메일 입력" required>
            </div>

            <button type="submit">등록</button>
        </form>
    </section>

    <section class="card">
        <h2>회원 목록</h2>

        <table>
            <thead>
            <tr>
                <th>ID</th>
                <th>이름</th>
                <th>이메일</th>
                <th>관리</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="member : ${members}">
                <td th:text="${member.id}">1</td>
                <td th:text="${member.name}">홍길동</td>
                <td th:text="${member.email}">hong@test.com</td>
                <td>
                    <a th:href="@{/members/{id}/edit(id=${member.id})}">수정</a>

                    <form th:action="@{/members/{id}/delete(id=${member.id})}" method="post" class="inline-form">
                        <button type="submit" class="delete-button">삭제</button>
                    </form>
                </td>
            </tr>
            </tbody>
        </table>
    </section>
</div>
</body>
</html>


Writing /content/spring-lab/simple-crud/src/main/resources/templates/members.html


In [13]:
%%writefile /content/spring-lab/simple-crud/src/main/resources/templates/edit.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>회원 수정</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="container">
    <h1>회원 수정</h1>

    <section class="card">
        <form th:action="@{/members/{id}/edit(id=${member.id})}" method="post" th:object="${memberForm}">
            <div class="form-row">
                <label>이름</label>
                <input type="text" th:field="*{name}" required>
            </div>

            <div class="form-row">
                <label>이메일</label>
                <input type="email" th:field="*{email}" required>
            </div>

            <button type="submit">수정 완료</button>
            <a href="/members" class="cancel-link">목록으로</a>
        </form>
    </section>
</div>
</body>
</html>


Writing /content/spring-lab/simple-crud/src/main/resources/templates/edit.html


In [14]:
%%writefile /content/spring-lab/simple-crud/src/main/resources/static/style.css

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #f5f6f8;
    color: #222;
}

.container {
    width: 900px;
    margin: 40px auto;
}

h1 {
    margin-bottom: 24px;
}

.card {
    background: #ffffff;
    padding: 24px;
    margin-bottom: 24px;
    border: 1px solid #ddd;
    border-radius: 10px;
}

.form-row {
    margin-bottom: 14px;
}

label {
    display: block;
    margin-bottom: 6px;
    font-weight: bold;
}

input {
    width: 100%;
    padding: 10px;
    border: 1px solid #ccc;
    border-radius: 6px;
}

button {
    padding: 9px 16px;
    border: 0;
    border-radius: 6px;
    background: #222;
    color: white;
    cursor: pointer;
}

.delete-button {
    background: #c0392b;
}

table {
    width: 100%;
    border-collapse: collapse;
}

th,
td {
    padding: 12px;
    border-bottom: 1px solid #ddd;
    text-align: left;
}

.inline-form {
    display: inline;
    margin-left: 8px;
}

a {
    color: #1f5eff;
    text-decoration: none;
}

.cancel-link {
    margin-left: 10px;
}

Writing /content/spring-lab/simple-crud/src/main/resources/static/style.css
